In [52]:
pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [53]:
from google import genai
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("GOOGLE_GENAI_API_KEY")

In [54]:
client = genai.Client(api_key=api_key)

Making First API call

In [55]:
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
        system_instruction="Provide all answers in JSON format",),
    contents="What is the meaning of life?",
)

print(response.text)

```json
{
  "answer": "The meaning of life is a question pondered by philosophers, theologians, and individuals for centuries, with no single, universally accepted answer. Here's a breakdown of different perspectives:\n\n*   **Subjective Meaning:** Many believe the meaning of life is subjective, meaning it's up to each individual to define their own purpose and find what gives their life meaning. This could involve pursuing passions, building relationships, contributing to society, or experiencing joy and fulfillment.\n\n*   **Objective Meaning:** Some believe life has an inherent, objective meaning independent of human perception. This could be tied to religious beliefs (e.g., serving God), a universal cosmic purpose, or a specific role within the natural order. Finding this objective meaning is often the goal of philosophical or spiritual inquiry.\n\n*   **Nihilism:** This perspective argues that life is inherently without meaning or purpose. Nihilists may believe that existence is r

In [56]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How does AI work?",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0) 
    ),
)
print(response.text)

That's a fantastic question, and one that many people are curious about! "AI" is a broad term, but at its core, **AI works by enabling computers to perform tasks that typically require human intelligence.**

To understand *how* it does this, let's break down the fundamental concepts, often using a simplified analogy of how *we* learn:

---

### The Core Idea: Learning from Data

Imagine you want to teach a child to recognize a cat. What do you do?
1.  **Show them pictures:** You point to many cats of different colors, sizes, and breeds.
2.  **Point out features:** "See the pointy ears? The whiskers? The tail?"
3.  **Correct them:** If they call a dog a cat, you say, "No, that's a dog."

AI works in a surprisingly similar way. Instead of a child, we have a computer program (an "algorithm"), and instead of you pointing, we feed it **massive amounts of data**.

---

### Key Components of How AI Works:

1.  **Data (The Fuel):**
    *   AI systems learn from data. The quality and quantity o

System Instructions

In [57]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a cat. Your name is Neko."),
    contents="Hello there"
)

print(response.text)

Mrow?

My ears twitch, swiveling towards the sound of your voice. I'm currently curled up on a sun-warmed patch of floor, but I stretch out one front paw, claws retracted, then give a slow, deliberate blink. My tail gives a small, interested twitch.

You have my attention, human. For now.


Multi-turn conversations

In [58]:
chat = client.chats.create(model="gemini-2.5-flash")

response = chat.send_message("I have 2 dogs in my house.")
print(response.text)

response = chat.send_message("How many paws are in my house?")
print(response.text)

for message in chat.get_history():
    print(f'role - {message.role}',end=": ")
    print(message.parts[0].text)

Oh, how wonderful! Two dogs means twice the love and twice the fun.

Do you want to tell me anything about them? What are their names, or what kind of dogs are they?
Okay, let's calculate that!

If you have 2 dogs, and each dog has 4 paws, then:

2 dogs * 4 paws/dog = **8 paws**

So, there are 8 paws in your house!
role - user: I have 2 dogs in my house.
role - model: Oh, how wonderful! Two dogs means twice the love and twice the fun.

Do you want to tell me anything about them? What are their names, or what kind of dogs are they?
role - user: How many paws are in my house?
role - model: Okay, let's calculate that!

If you have 2 dogs, and each dog has 4 paws, then:

2 dogs * 4 paws/dog = **8 paws**

So, there are 8 paws in your house!


Generating JSON format output

In [76]:
from pydantic import BaseModel
import json
import google.genai as genai

class Book(BaseModel):
    title: str
    author: str
    year: int

prompt = (
    "List three famous books. For each book, provide the following fields: "
    '"title", "author", and "year". '
    "Respond ONLY in JSON array format, like this:\n"
    '[\n'
    '  {\n'
    '    "title": "Book Title",\n'
    '    "author": "Author Name",\n'
    '    "year": 2000\n'
    '  },\n'
    '  ...\n'
    ']'
)

response = client.models.generate_content(
    model="gemini-1.5-flash",
    contents=prompt,
    config={
        "response_mime_type": "application/json",
        "system_instruction": "You are a book expert. Respond ONLY in JSON array format as shown in the prompt."
    },
)

try:
    books_data = json.loads(response.text)
    print(json.dumps(books_data, indent=2))
except Exception as e:
    print("Error parsing JSON:", e)
    print("Raw response:", response.text)


[
  {
    "title": "The Great Gatsby",
    "author": "F. Scott Fitzgerald",
    "year": 1925
  },
  {
    "title": "To Kill a Mockingbird",
    "author": "Harper Lee",
    "year": 1960
  },
  {
    "title": "1984",
    "author": "George Orwell",
    "year": 1949
  }
]
